# JiT-RCDM Training on Google Colab

Representation-Conditioned Diffusion Model — DinoV3 ViT-S/16 + JiT ViT + Flow Matching  
Dataset: Messidor-2 retinal fundus images (224×224)

## Before you start
1. **Runtime → Change runtime type → A100 GPU**
2. Upload the following to **Google Drive** under `MyDrive/jit_rcdm/`:
   - `train_reps.pt` — precomputed DinoV3 representations (~3 MB for 972 images)
   - `dinov3_vits16_tmp/` — the DinoV3 checkpoint folder (`config.json` + `model.safetensors`)
   - *(Optional)* `MESSIDOR2/` — raw images, only needed to re-run precompute_reps
3. Run cells top to bottom

## Workflow after code changes
Edit locally → `git push` → re-run **Cell 3** in Colab (`git pull`). No restart needed.

## 1 — Check GPU

In [1]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM    : {mem:.1f} GB")
    if mem >= 70:
        print("✓ A100/H100 — use batch_size=256")
    elif mem >= 30:
        print("✓ A100 40 GB — use batch_size=128")
    else:
        print("⚠ T4/V100 — use batch_size=32")
else:
    print("❌ No GPU — go to Runtime → Change runtime type → GPU")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-40GB
VRAM    : 42.4 GB
✓ A100 40 GB — use batch_size=128


## 2 — Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/jit_rcdm"
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"Drive mounted. Project folder: {DRIVE_DIR}")
print("Contents:")
for f in sorted(os.listdir(DRIVE_DIR)):
    size = os.path.getsize(os.path.join(DRIVE_DIR, f)) if os.path.isfile(os.path.join(DRIVE_DIR, f)) else 0
    tag  = f"({size/1e6:.1f} MB)" if size > 0 else "(folder)"
    print(f"  {f} {tag}")

Mounted at /content/drive
Drive mounted. Project folder: /content/drive/MyDrive/jit_rcdm
Contents:
  dinov3_vits16_tmp (folder)
  test_images (folder)
  train_reps.pt (1.6 MB)


## 3 — Clone / pull JiT-RCDM code

In [3]:
import os, sys

REPO_DIR    = "/content/jit_rcdm"
REPO_URL    = "https://github.com/SeverinLe/master_implementation.git"
BRANCH      = "claude/silly-faraday-d8512b"   # JiT-RCDM branch

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull
else:
    print(f"Cloning branch {BRANCH} ...")
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}

# Add repo root to Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%cd {REPO_DIR}
print("\nContents:")
!ls -1

Cloning branch claude/silly-faraday-d8512b ...
Cloning into '/content/jit_rcdm'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 73 (delta 15), reused 67 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 93.09 KiB | 3.32 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/jit_rcdm

Contents:
COLAB_SETUP_GUIDE.md
colab_training.ipynb
guided_diffusion
prepare_for_colab.sh
rcdm
sanitycheckSL.py
scripts
test_encoder.py


## 4 — Install dependencies

Colab ships with PyTorch ≥ 2.4 which is required for `transformers` to load DinoV3.

In [4]:
!pip install -q transformers safetensors wandb
print("✓ Dependencies installed")

# Verify transformers can actually load (requires PyTorch >= 2.4)
from transformers import AutoModel
print("✓ transformers import OK")

✓ Dependencies installed
✓ transformers import OK


## 5 — Copy DinoV3 checkpoint from Drive

In [5]:
import shutil, os

SRC = "/content/drive/MyDrive/jit_rcdm/dinov3_vits16_tmp"
DST = "/content/jit_rcdm/checkpoints/dinov3_vits16_tmp"

assert os.path.isdir(SRC), (
    f"DinoV3 checkpoint not found at {SRC}.\n"
    "Upload the dinov3_vits16_tmp/ folder to Google Drive at MyDrive/jit_rcdm/"
)

if not os.path.isdir(DST):
    shutil.copytree(SRC, DST)
    print(f"✓ Copied DinoV3 checkpoint → {DST}")
else:
    print(f"✓ DinoV3 checkpoint already at {DST}")

import json
cfg = json.load(open(os.path.join(DST, "config.json")))
print(f"  hidden_size      : {cfg['hidden_size']}  (expect 384)")
print(f"  use_gated_mlp    : {cfg['use_gated_mlp']}  (must be false)")
print(f"  num_register_tokens: {cfg['num_register_tokens']}")
assert cfg['use_gated_mlp'] == False, "use_gated_mlp must be false — edit config.json"
assert cfg['hidden_size'] == 384,    "Unexpected encoder hidden size"
print("✓ Config OK")

✓ Copied DinoV3 checkpoint → /content/jit_rcdm/checkpoints/dinov3_vits16_tmp
  hidden_size      : 384  (expect 384)
  use_gated_mlp    : False  (must be false)
  num_register_tokens: 4
✓ Config OK


## 6 — Copy training representations from Drive

`train_reps.pt` was precomputed locally with the fixed encoder (`use_gated_mlp=false`).  
Upload it to Drive at `MyDrive/jit_rcdm/train_reps.pt` before running this cell.

In [6]:
import shutil, os
import torch

SRC = "/content/drive/MyDrive/jit_rcdm/train_reps.pt"
DST = "/content/jit_rcdm/data/messidor2/train_reps.pt"

assert os.path.exists(SRC), (
    f"train_reps.pt not found at {SRC}.\n"
    "Run precompute_reps.py locally first, then upload to Google Drive."
)

os.makedirs(os.path.dirname(DST), exist_ok=True)
if not os.path.exists(DST):
    print("Copying train_reps.pt from Drive...")
    shutil.copy2(SRC, DST)
else:
    print("train_reps.pt already present")

data = torch.load(DST, map_location="cpu")
reps = data["reps"]
print(f"✓ Loaded: {len(data['paths'])} images, reps shape {tuple(reps.shape)}")
assert reps.shape[1] == 384, f"Expected 384-dim reps, got {reps.shape[1]}"
print(f"  Sample L2 norms (first 5): {reps[:5].norm(dim=1).tolist()}")
print("✓ Representations OK")

Copying train_reps.pt from Drive...
✓ Loaded: 972 images, reps shape (972, 384)
  Sample L2 norms (first 5): [9.964570999145508, 9.961363792419434, 9.962718963623047, 10.104928970336914, 9.93552017211914]
✓ Representations OK


## 7 — Verify full pipeline imports

In [7]:
import sys, os
sys.path.insert(0, "/content/jit_rcdm")

from rcdm.encoder      import load_encoder, build_transform, DINOV3_CHECKPOINT
from rcdm.jit          import JiT_S_16, FlowMatching, create_jit_model
from rcdm.dataset      import RepresentationDataset
from rcdm.conditioning import RMSNorm, AdaLNZero, ConditioningProjector
import torch

print("✓ All imports OK")

# Quick model smoke-test (CPU, no encoder needed)
m = JiT_S_16(image_size=224, h_dim=384)
m.eval()
z = torch.randn(2, 3, 224, 224)
t = torch.rand(2)
h = torch.randn(2, 384)
with torch.no_grad():
    out = m(z, t, h)
assert out.shape == (2, 3, 224, 224), f"Unexpected output shape: {out.shape}"
n_params = sum(p.numel() for p in m.parameters())
print(f"✓ JiT_S_16 forward pass OK — output {tuple(out.shape)}, {n_params/1e6:.1f}M params")
print(f"✓ null_h parameter: shape {tuple(m.null_h.shape)}, requires_grad={m.null_h.requires_grad}")
print(f"✓ freqs_cis buffer:  shape {tuple(m.freqs_cis.shape)}, dtype={m.freqs_cis.dtype}")

ImportError: cannot import name 'DINOV3_CHECKPOINT' from 'rcdm.encoder' (/content/jit_rcdm/rcdm/encoder.py)

## 8 — Train

Adjust `--batch_size` based on your GPU:  
- A100 80 GB → **256**  
- A100 40 GB → **128**  
- V100 / T4 → **32–64**

Checkpoints are saved to Drive and survive session restarts.

In [ ]:
import torch
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
batch  = 256 if mem_gb >= 70 else (128 if mem_gb >= 38 else 32)
print(f"GPU VRAM: {mem_gb:.0f} GB → batch_size={batch}")

In [ ]:
import os
CKPT_DIR = "/content/drive/MyDrive/jit_rcdm/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

!python scripts/train.py \
    --model         S16 \
    --reps_file     data/messidor2/train_reps.pt \
    --save_dir      {CKPT_DIR} \
    --image_size    224 \
    --batch_size    {batch} \
    --grad_accum    1 \
    --lr            1e-4 \
    --warmup_steps  2000 \
    --total_steps   100000 \
    --save_interval 5000 \
    --log_interval  100 \
    --cfg_dropout   0.1 \
    --ema_decay     0.9999 \
    --wandb_project jit-rcdm \
    --wandb_run_name S16-100k-A100 \
    --device        cuda

## 8b — Resume after disconnection

Re-run cells 1–7, then run this cell. Replace `STEPXXXXXXX` with the latest checkpoint step.

In [ ]:
import os, glob
CKPT_DIR = "/content/drive/MyDrive/jit_rcdm/checkpoints"

# Auto-find the latest checkpoint
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "jit_rcdm_step*.pt")))
if ckpts:
    latest = ckpts[-1]
    print(f"Resuming from: {latest}")
    !python scripts/train.py \
        --model         S16 \
        --reps_file     data/messidor2/train_reps.pt \
        --save_dir      {CKPT_DIR} \
        --image_size    224 \
        --batch_size    {batch} \
        --grad_accum    1 \
        --lr            1e-4 \
        --warmup_steps  2000 \
        --total_steps   200000 \
        --save_interval 10000 \
        --log_interval  100 \
        --cfg_dropout   0.1 \
        --ema_decay     0.9999 \
        --resume        {latest} \
        --wandb_project jit-rcdm \
        --wandb_run_name S16-200k-A100 \
        --device        cuda
else:
    print("No checkpoint found in", CKPT_DIR)

## 9 — Sample (after training)

Use `--cfg_scale 1.0` before step 15 k, then step up to 1.5 → 2.0 → 3.0.

In [ ]:
import os, glob
CKPT_DIR   = "/content/drive/MyDrive/jit_rcdm/checkpoints"
SAMPLE_DIR = "/content/drive/MyDrive/jit_rcdm/samples"
os.makedirs(SAMPLE_DIR, exist_ok=True)

# Auto-find latest checkpoint
ckpts  = sorted(glob.glob(os.path.join(CKPT_DIR, "jit_rcdm_step*.pt")))
latest = ckpts[-1] if ckpts else os.path.join(CKPT_DIR, "jit_rcdm_final.pt")
print(f"Sampling from: {latest}")

# Upload a few test images to Drive at MyDrive/jit_rcdm/test_images/ first
TEST_IMAGES = sorted(glob.glob("/content/drive/MyDrive/jit_rcdm/test_images/*.png"))
assert TEST_IMAGES, "Upload test images to MyDrive/jit_rcdm/test_images/"
cond_args = " ".join(TEST_IMAGES)

!python scripts/sampling.py \
    --checkpoint  {latest} \
    --cond_images {cond_args} \
    --out_dir     {SAMPLE_DIR} \
    --n_samples   4 \
    --num_steps   50 \
    --cfg_scale   1.0 \
    --wandb_project jit-rcdm \
    --wandb_run_name S16-samples \
    --device      cuda

## 10 — Monitor

In [ ]:
!nvidia-smi
!ls -lh /content/drive/MyDrive/jit_rcdm/checkpoints/

---

## Reference

| Step | Expected time (A100 80 GB, batch=256) |
|------|----------------------------------------|
| 50 k steps  | ~25 min |
| 100 k steps | ~50 min |
| 200 k steps | ~100 min |

### CFG scale schedule

| Steps trained | Recommended `--cfg_scale` |
|---------------|---------------------------|
| < 15 k        | 1.0 (pure conditional, no extrapolation) |
| 15–30 k       | 1.5 |
| 30–50 k       | 2.0 |
| > 50 k        | 2.0–3.0 |

### Google Drive layout expected

```
MyDrive/jit_rcdm/
    dinov3_vits16_tmp/        ← DinoV3 checkpoint folder
        config.json
        model.safetensors
    train_reps.pt             ← precomputed representations (run locally first)
    test_images/              ← a few .png fundus images for sampling
    checkpoints/              ← auto-created by training
    samples/                  ← auto-created by sampling
```